# Decoder

In [ ]:
#TODO Add the decoder image here.

In [ ]:
import torch
import torch.nn as nn
from typing import Optional

In [ ]:
class DecoderLayer(nn.Module):
    def __init__(self, config):
        """
        A single decoder layer of the Llama Decoder Nx stack.
        """
        super().__init__()

        self.self_attn = GroupedQueryAttention(config)  # Handle GQA and RoPE
        self.ffn = SwiGLUFeedForward(config)

        # Pre-normalization: Must have separate RMSNorm layers, so each can have a different learned weights
        self.attention_norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)
        self.ffn_norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)

    def forward(
        self,
        x: torch.Tensor,
        freqs_cis: torch.Tensor,
        causal_mask: Optional[torch.Tensor] = None,
        kv_cache: Optional[tuple] = None,
    ):
        """
        Args:
            x: The input tensor of shape (batch_size, seq_len, hidden_size).
            freqs_cis: The precomputed tensor for RoPE.
            mask: causal mask to prevent attention to future tokens.
            kv_cache: Inference only, Tuple of (key, value) for autoregressive inference.
        """

        # 1. Apply RMS Norm
        # 2. Apply GQA with RoPE
        # 3 Add first residual connection
        i = x + self.self_attn(
            x=self.attention_norm(x),
            freqs_cis=freqs_cis,
            mask=causal_mask,
            kv_cache=kv_cache,
        )
        
        # 4 Apply RMS Norm
        # 5 Apply Feed Forward SwiGLU
        # 6 Add second residual connection
        out = i + self.ffn(self.ffn_norm(i))
        return out

In [ ]:
class Decoder(nn.Module):
    def __init__(self, layer:DecoderLayer, N):
        super().__init__()

        # Make the Nx decoder layers
        self.layers = nn.ModuleList([DecoderLayer(config) for _ in range(config.num_hidden_layers)])
        self.norm = RMSNorm(layer.d_model)

        # The final normalization before the output linear 
        self.final_norm = RMSNorm(config.hidden_size, eps=config.rms_norm_eps)